## Statistical analysis of potential improvements at different training sizes for indel datasets

Compares the ESM2 representation against the three baseline
representations (physicochemical descriptors, Morgan fingerprints and
character n-grams) with a repeated-measures Tukey HSD test, using the
random seed as the repeated measure.


### Read data


In [4]:
import pandas as pd
from model_comparison import rm_tukey_hsd

# ------------------------------------------------------------------
# One row per (dataset, representation, seed, train size) simulation run.
# Produced by aggregate_simulation_results.py as simulation_summary_all.csv.
# ------------------------------------------------------------------
FAMILY = "indels"
SPLIT = "Random"          # how the train/test split was made for this family

import os


def find_results_root(*candidates):
    """Return the first results folder that exists.

    The SLURM jobs write to simulate_experiments/simulation_output/; the
    simulation_outputs/ folder is only a copy (e.g. results moved off the
    cluster), so the jobs' own output is checked first and a stale copy can
    never shadow fresh results.
    """
    for c in candidates:
        if os.path.isdir(c):
            print(f"Reading results from {os.path.abspath(c)}")
            return c
    raise FileNotFoundError(
        "No benchmark results folder found. Looked in:\n  "
        + "\n  ".join(os.path.abspath(c) for c in candidates)
    )

SIM_ROOT = find_results_root(
    "../simulate_experiments/simulation_output",
    "../simulation_output",
    "../simulation_outputs",
)

# Tables below are written here.
os.makedirs("results", exist_ok=True)
df_input = pd.read_csv(os.path.join(SIM_ROOT, FAMILY, "simulation_summary_all.csv"))

# Make name shorter
df_input.loc[df_input["Model"] == "GaussianProcessRegressor", "Model"] = "GPR"

# ------------------------------------------------------------------
# Label each run by the sequence representation it used. The benchmark
# compares one foundation model against three baselines.
# ------------------------------------------------------------------
REPRESENTATION_LABELS = {
    "esm":      "GPR + ESM2",
    "physchem": "GPR + Physchem",
    "morgan":   "GPR + Morgan",
    "ngram":    "GPR + N-gram",
}

#: The representation every other one is compared against.
REFERENCE = "GPR + ESM2"

unknown = set(df_input["Representation"].unique()) - set(REPRESENTATION_LABELS)
if unknown:
    raise ValueError(f"Unexpected representations in the results: {sorted(unknown)}")

df_input["Model Label"] = df_input["Representation"].map(REPRESENTATION_LABELS)

df_input = df_input[df_input["Model Label"].isin(REPRESENTATION_LABELS.values())]

df_input["method"] = df_input["Model Label"]
df_input["cv_cycle"] = df_input["Seed"]
df_input["split"] = SPLIT

datasets = [
    "PKN1_HUMAN_Tsuboyama_2023_1URF_indels",
    "RPC1_BP434_Tsuboyama_2023_1R69_indels",
    "POLG_PESV_Tsuboyama_2023_2MXD_indels",
    "SDA_BACSU_Tsuboyama_2023_1PV0_indels",
    "RD23A_HUMAN_Tsuboyama_2023_1IFY_indels",
    "MAFG_MOUSE_Tsuboyama_2023_1K1V_indels",
    "SQSTM_MOUSE_Tsuboyama_2023_2RRU_indels",
    "VG08_BPP22_Tsuboyama_2023_2GP8_indels",
    "YNZC_BACSU_Tsuboyama_2023_2JVD_indels",
    "PIN1_HUMAN_Tsuboyama_2023_1I6C_indels",
]

train_sizes = sorted(df_input["Train Size"].unique())

df_all = {data: {} for data in datasets}
for data in datasets:
    for size in train_sizes:
        df_all[data][size] = df_input[
            (df_input["Train Size"] == size) & (df_input["Dataset"] == data)
        ]

# Defining column names for metrics
metric_ls = [
    "Test MSE",
    "Test RMSE",
    "Test MAE",
    "Test R2",
    "Test Spearman Correlation",
    "Top 20 Accuracy",
]

print(f"Representations found: {sorted(df_input['Model Label'].unique())}")
print(f"Train sizes: {train_sizes}")
print(f"Datasets: {len(datasets)}")


Reading results from /Users/leoan/Documents/GitHub/SCARSE-repro/code/benchmark/simulate_experiments/simulation_output
Representations found: ['GPR + ESM2', 'GPR + Morgan', 'GPR + N-gram', 'GPR + Physchem']
Train sizes: [np.int64(20), np.int64(40), np.int64(60), np.int64(80)]
Datasets: 10


### Check the design is complete


In [5]:
# ------------------------------------------------------------------
# The repeated-measures test needs a complete design: every representation
# present for every seed. A missing simulation run would otherwise make
# rm_anova fail, or silently change the residual degrees of freedom.
# ------------------------------------------------------------------
incomplete = []
for data in datasets:
    for size in train_sizes:
        d = df_all[data][size]
        if len(d) == 0:
            incomplete.append((data, size, "no runs"))
            continue
        counts = d.pivot_table(index="cv_cycle", columns="Model Label",
                               values="Test MSE", aggfunc="size")
        if counts.isna().any().any() or counts.shape[1] < len(REPRESENTATION_LABELS):
            missing = int(counts.isna().sum().sum())
            incomplete.append((data, size,
                               f"{counts.shape[1]}/{len(REPRESENTATION_LABELS)} representations, "
                               f"{missing} missing seed cells"))

if incomplete:
    print(f"WARNING: {len(incomplete)} (dataset, train size) cells are incomplete "
          "and will be skipped:\n")
    for data, size, why in incomplete[:20]:
        print(f"  {data}  train size {size}: {why}")
    if len(incomplete) > 20:
        print(f"  ... and {len(incomplete) - 20} more")
else:
    print("Design is complete: every representation present for every seed.")

skip = {(data, size) for data, size, _ in incomplete}


Design is complete: every representation present for every seed.


### Evaluation summary across datasets per training size


Takes the mean p-value and mean size of difference across datasets per training size, for each pair of representations


In [6]:
# ------------------------------------------------------------------
# With four representations there are six pairwise comparisons per
# (dataset, train size, metric), so every comparison is recorded with the
# pair it belongs to rather than only the first row of the Tukey table.
# ------------------------------------------------------------------
rows = []
nan_cells = []   # cells where a metric was undefined for some runs

for metric in metric_ls:
    for size in train_sizes:
        for data in datasets:
            if (data, size) in skip:
                continue

            df_all_temp = df_all[data][size]

            # A metric can be undefined for a run (Spearman is NaN when a model
            # predicts the same value for every test sequence). The repeated-measures
            # ANOVA only uses seeds that are complete across all groups; do that step
            # here, so it is reported and the Tukey means use the same seeds.
            wide = df_all_temp.pivot_table(index="cv_cycle", columns="Model Label",
                                       values=metric, dropna=False)
            undefined = [c for c in wide.columns if wide[c].isna().all()]
            wide = wide.drop(columns=undefined).dropna()
            n_seeds = df_all_temp["cv_cycle"].nunique()
            if wide.shape[1] < 2 or wide.shape[0] < 2:
                nan_cells.append((metric, (data, size), "not tested", undefined,
                    wide.shape[0] if wide.shape[1] else 0, n_seeds))
                continue
            if undefined or wide.shape[0] < n_seeds:
                nan_cells.append((metric, (data, size), "reduced", undefined, wide.shape[0], n_seeds))
            df_all_temp = df_all_temp[df_all_temp["Model Label"].isin(wide.columns)
                          & df_all_temp["cv_cycle"].isin(wide.index)]

            results = rm_tukey_hsd(df_all_temp, metric, "Model Label")[0]

            for _, r in results.iterrows():
                rows.append({
                    "Train Size": size,
                    "Dataset": data,
                    "Metric": metric,
                    "group1": r["group1"],
                    "group2": r["group2"],
                    "p-adj": r["p-adj"],
                    "effect size": r["meandiff"],
                })

df_results_all = pd.DataFrame(rows)

if nan_cells:
    n_skip = sum(1 for r in nan_cells if r[2] == "not tested")
    print(f"Undefined (NaN) metric values in {len(nan_cells)} cells "
          f"({n_skip} could not be tested, {len(nan_cells) - n_skip} tested on fewer seeds).")
    print("A run's metric is NaN when it cannot be computed - Spearman is undefined")
    print("when a model predicts the same value for every test sequence.\n")
    for metric, where, status, undefined, kept, total in nan_cells:
        what = (f"undefined for {', '.join(undefined)}; " if undefined else "")
        print(f"  {metric:26s} {where[0]}  train size {where[1]}: {status} - {what}{kept}/{total} complete seeds")
    print()

# Mean p-value and mean size of difference across datasets, per training size,
# metric and pair of representations.
df_results = (
    df_results_all
    .groupby(["Train Size", "Metric", "group1", "group2"], as_index=False)
    .agg(**{
        "p-adj mean value": ("p-adj", "mean"),
        "p-adj std value": ("p-adj", "std"),
        "effect size mean value": ("effect size", "mean"),
        "effect size std value": ("effect size", "std"),
        "n datasets": ("p-adj", "size"),
    })
    .sort_values(["Train Size", "Metric", "group1", "group2"])
    .reset_index(drop=True)
)

# The comparisons of interest: the foundation model against each baseline.
is_ref = (df_results["group1"] == REFERENCE) | (df_results["group2"] == REFERENCE)
df_results_reference = df_results[is_ref].reset_index(drop=True)

df_results.to_csv(f"results/{FAMILY}_results_summary.csv", index=False)
df_results_all.to_csv(f"results/{FAMILY}_results.csv", index=False)
df_results_reference.to_csv(f"results/{FAMILY}_results_summary_vs_reference.csv", index=False)

print(f"Wrote results/{FAMILY}_results.csv                        "
      f"({len(df_results_all)} comparisons)")
print(f"Wrote results/{FAMILY}_results_summary.csv                "
      f"({len(df_results)} rows)")
print(f"Wrote results/{FAMILY}_results_summary_vs_reference.csv   "
      f"({len(df_results_reference)} rows)")

df_results_reference.head(12)


/Users/leoan/Documents/GitHub/SCARSE-repro/.venv/lib/python3.12/site-packages/pingouin/parametric.py:575: RuntimeWarning: invalid value encountered in scalar divide
  fval = ms_with / ms_reswith
/Users/leoan/Documents/GitHub/SCARSE-repro/.venv/lib/python3.12/site-packages/pingouin/parametric.py:585: RuntimeWarning: invalid value encountered in scalar divide
  ef = ss_with / (ss_with + ss_resall)
/Users/leoan/Documents/GitHub/SCARSE-repro/.venv/lib/python3.12/site-packages/pingouin/distribution.py:742: RuntimeWarning: invalid value encountered in scalar divide
  eps = np.min([num / den, 1])
/Users/leoan/Documents/GitHub/SCARSE-repro/code/benchmark/simulate_experiments/statistics/model_comparison.py:104: RuntimeWarning: invalid value encountered in scalar divide
  studentized_range = np.abs(mean_diff) / tukey_se
/Users/leoan/Documents/GitHub/SCARSE-repro/.venv/lib/python3.12/site-packages/scipy/optimize/_optimize.py:2427: OptimizeWarning: 
NaN result encountered.
  _endprint(x, flag, fva

Undefined (NaN) metric values in 39 cells (3 could not be tested, 36 tested on fewer seeds).
A run's metric is NaN when it cannot be computed - Spearman is undefined
when a model predicts the same value for every test sequence.

  Test Spearman Correlation  PKN1_HUMAN_Tsuboyama_2023_1URF_indels  train size 20: reduced - undefined for GPR + Physchem; 10/10 complete seeds
  Test Spearman Correlation  RPC1_BP434_Tsuboyama_2023_1R69_indels  train size 20: not tested - 0/10 complete seeds
  Test Spearman Correlation  POLG_PESV_Tsuboyama_2023_2MXD_indels  train size 20: reduced - 3/10 complete seeds
  Test Spearman Correlation  SDA_BACSU_Tsuboyama_2023_1PV0_indels  train size 20: reduced - 2/10 complete seeds
  Test Spearman Correlation  RD23A_HUMAN_Tsuboyama_2023_1IFY_indels  train size 20: reduced - undefined for GPR + Physchem; 9/10 complete seeds
  Test Spearman Correlation  MAFG_MOUSE_Tsuboyama_2023_1K1V_indels  train size 20: not tested - 1/10 complete seeds
  Test Spearman Correlation

,Train Size,Metric,group1,group2,p-adj mean value,p-adj std value,effect size mean value,effect size std value,n datasets
0,20,Test MAE,GPR + ESM2,GPR + Morgan,0.013171,0.032335,-0.371491,0.178647,10
1,20,Test MAE,GPR + ESM2,GPR + N-gram,0.001062,0.000197,-0.355917,0.140856,10
2,20,Test MAE,GPR + ESM2,GPR + Physchem,0.004941,0.009709,-0.395468,0.193430,10
3,20,Test MSE,GPR + ESM2,GPR + Morgan,0.026088,0.071049,-0.666458,0.348877,10
4,20,Test MSE,GPR + ESM2,GPR + N-gram,0.007888,0.011160,-0.584580,0.300779,10
5,20,Test MSE,GPR + ESM2,GPR + Physchem,0.025333,0.045479,-0.659016,0.355235,10
6,20,Test R2,GPR + ESM2,GPR + Morgan,0.018977,0.038550,0.718869,0.190280,10
7,20,Test R2,GPR + ESM2,GPR + N-gram,0.025648,0.070414,0.630508,0.139439,10
8,20,Test R2,GPR + ESM2,GPR + Physchem,0.039654,0.090889,0.689749,0.205628,10
9,20,Test RMSE,GPR + ESM2,GPR + Morgan,0.005504,0.013060,-0.412290,0.180841,10
